# Multi-Trace U-Net 22M · Modal · resume / diagnostics / Hugging Face

Tham chiếu `Untitled0.ipynb`: cùng U-Net, shared u head, K=1..5,
interval loss và EMA; tăng backbone **5.946.579 → 22.002.655 parameters**.
Base channels 44 → 84, condition dimension 160 → 356.

Đây là mở rộng Multi-Trace 1–5 NFE; objective one-step nguyên bản trong
proposal chỉ train hai trace r=0/r=t. Config và lý do scale ở SCALING.md.

**Trước khi Run all:** chọn 1 GPU (H200 đang dùng được), CPU 2 cores,
RAM 8 GiB; attach Volume `imf-training` tại `/mnt/imf-training`;
attach Secret `HF_TOKEN` quyền write, thêm `GITHUB_TOKEN` nếu repo private.
Không đặt token trực tiếp trong notebook.

Effective batch 512 = micro-batch 128 × accumulation 4. Giữ 100.000
optimizer updates, warmup 5.000, LR 1e-4 → 1e-5, clipping 1.0, EMA .9999.
Đổi effective batch thì số updates/warmup/EMA tự tính theo image budget
51.2M/2.56M ảnh. LR và clipping không nhân theo số parameters.

Mặc định **chạy thử đến update 1000, lưu checkpoint rồi tạm dừng** để đo
throughput/ETA trước khi dùng tiếp credit. Đặt `STOP_AFTER_UPDATES=None`
và chạy lại pipeline để resume đến hết schedule, đánh giá FID 1–5 NFE
rồi tự upload EMA model và diagnostics lên Hugging Face.
Giới hạn thử không đổi LR schedule. 100k updates + FID có thể vượt $30;
notebook không tự suy ra chi phí từ tên GPU.
Dùng RUN_NAME mới; không resume checkpoint DiT từ notebook cũ.


In [ ]:
from pathlib import Path

REPO_URL = "https://github.com/namdeptraivcd/IMF.git"
REPO_REF = "codex/modal-training"
VOLUME_ROOT = Path("/mnt/imf-training")
RUN_NAME = "multitrace_unet22m_v1"
BATCH_SIZE = 128  # physical image microbatch; joint edge/diag forward uses 256 inputs
ACCUMULATION_STEPS = 4
NUM_WORKERS = 2
STOP_AFTER_UPDATES = 1_000  # None: train full image budget and auto upload
LOG_EVERY = 50
PLOT_EVERY = 500
VALIDATION_EVERY = 1_000
VALIDATION_BATCHES = 5  # fixed images and all intervals for each K=1..5
SAMPLE_EVERY = 1_000
CHECKPOINT_EVERY = 1_000
KEEP_LAST_CHECKPOINTS = 3
REFRESH_SECONDS = 10
FID_NUM_GENERATED = 10_000
FID_BATCH_SIZE = 128
HF_REPO_ID = None  # or "username/multitrace-imf-cifar10-unet22m"
HF_PRIVATE = True
AUTO_RESUME = True
RESUME_CHECKPOINT = None


## Clone code và pin commit

Clone từ GitHub, checkout REPO_REF. Khi resume, dùng Git commit từ
source_manifest.json của run trước. Kernel restart cần chạy lại notebook
từ đầu với cùng RUN_NAME/config. Không chạy hai kernel vào cùng run.


In [ ]:
import hashlib
import json
import os
import re
import shutil
import subprocess
import sys
import tempfile

# Modal exposes /mnt mounts as symlinks into /__modal/volumes.
# Validate the configured mount path without resolving that symlink.
if not VOLUME_ROOT.is_dir() or not Path(os.path.abspath(VOLUME_ROOT)).is_relative_to(Path("/mnt")):
    raise RuntimeError(f"Attach Volume tại {VOLUME_ROOT} trước khi chạy")
if not re.fullmatch(r"[A-Za-z0-9_-]+", RUN_NAME):
    raise ValueError("RUN_NAME chỉ dùng chữ, số, underscore và gạch ngang")
RUN_DIR = VOLUME_ROOT / "runs" / RUN_NAME
manifest_path = RUN_DIR / "source_manifest.json"
saved_manifest = json.loads(manifest_path.read_text()) if manifest_path.exists() else None
chosen_ref = saved_manifest["git_commit"] if saved_manifest and AUTO_RESUME else REPO_REF
SOURCE_DIR = Path("/tmp") / f"imf-repo-{RUN_NAME}"
git_env = os.environ.copy()
git_env["GIT_TERMINAL_PROMPT"] = "0"
if git_env.get("GITHUB_TOKEN"):
    askpass = Path("/tmp/imf-git-askpass.py")
    askpass.write_text("#!/usr/bin/env python3\nimport os,sys\n"
        "print('x-access-token' if 'Username' in sys.argv[1] else os.environ['GITHUB_TOKEN'])\n")
    askpass.chmod(0o700)
    git_env["GIT_ASKPASS"] = str(askpass)
if not (SOURCE_DIR / ".git").exists():
    subprocess.run(["git", "clone", "--filter=blob:none", "--no-checkout",
                    REPO_URL, str(SOURCE_DIR)], env=git_env, check=True)
remote = subprocess.check_output(["git", "-C", str(SOURCE_DIR), "remote", "get-url", "origin"], text=True).strip()
if remote != REPO_URL:
    raise ValueError("SOURCE_DIR chứa repo khác; restart kernel hoặc đổi RUN_NAME")
subprocess.run(["git", "-C", str(SOURCE_DIR), "fetch", "origin", chosen_ref], env=git_env, check=True)
subprocess.run(["git", "-C", str(SOURCE_DIR), "checkout", "--detach", "FETCH_HEAD"], env=git_env, check=True)
GIT_COMMIT = subprocess.check_output(["git", "-C", str(SOURCE_DIR), "rev-parse", "HEAD"], text=True).strip()
required = ["train.py", "monitoring.py", "hub.py", "sample.py", "models/dit.py", "models/__init__.py",
            "trace_imf.py", "data.py", "configs/__init__.py",
            "configs/cifar10_22m.py", "requirements-notebook.txt"]
required += ["models/unet.py", "multi_trace_imf.py", "objectives.py", "ema.py",
             "evaluation.py", "configs/multitrace_cifar10_22m.py"]
missing = [name for name in required if not (SOURCE_DIR / name).is_file()]
if missing:
    raise RuntimeError(f"Git ref thiếu code mới: {missing}. Push branch/ref triển khai rồi chạy lại.")
SOURCE_MANIFEST = {"repo_url": REPO_URL, "git_commit": GIT_COMMIT,
    "files": {name: hashlib.sha256((SOURCE_DIR / name).read_bytes()).hexdigest() for name in required}}
if saved_manifest and SOURCE_MANIFEST != saved_manifest:
    raise ValueError("Source manifest thay đổi; chọn RUN_NAME mới")
sys.path.insert(0, str(SOURCE_DIR))
print("Cloned IMF commit:", GIT_COMMIT)

for module_name in ("hub", "monitoring", "models", "objectives", "trace_imf", "multi_trace_imf"):
    loaded = sys.modules.get(module_name)
    if loaded and getattr(loaded, "__file__", None):
        if not Path(loaded.__file__).resolve().is_relative_to(SOURCE_DIR.resolve()):
            raise RuntimeError("Restart kernel trước khi đổi notebook/repo source để xóa module cache")


In [ ]:
if shutil.which("uv"):
    install_command = ["uv", "pip", "install", "--python", sys.executable,
                       "-r", str(SOURCE_DIR / "requirements-notebook.txt")]
else:
    install_command = [sys.executable, "-m", "pip", "install",
                       "-r", str(SOURCE_DIR / "requirements-notebook.txt")]
subprocess.run(install_command, check=True)


In [ ]:
import torch
import torchvision
from huggingface_hub import HfApi
from hub import prepare_repository

if not torch.cuda.is_available():
    raise RuntimeError("Chọn GPU trong Compute profile của Modal rồi restart kernel")
if min(ACCUMULATION_STEPS, FID_NUM_GENERATED, FID_BATCH_SIZE, BATCH_SIZE, LOG_EVERY, PLOT_EVERY, VALIDATION_EVERY,
       VALIDATION_BATCHES, SAMPLE_EVERY, CHECKPOINT_EVERY, KEEP_LAST_CHECKPOINTS,
       REFRESH_SECONDS) <= 0 or NUM_WORKERS < 0:
    raise ValueError("Số bước/batch/interval phải dương; workers không âm")
token = os.environ.get("HF_TOKEN")
if not token:
    raise RuntimeError("Attach Secret có HF_TOKEN quyền write rồi restart kernel")
account = HfApi(token=token).whoami()["name"]
REPO_ID = HF_REPO_ID or f"{account}/multitrace-imf-cifar10-unet22m"
hub_api = prepare_repository(REPO_ID, token, private=HF_PRIVATE)
del token
print("GPU:", torch.cuda.get_device_name(0))
print("torch / torchvision:", torch.__version__, torchvision.__version__)
print("Hub destination:", f"https://huggingface.co/{REPO_ID}", "private" if HF_PRIVATE else "public")

if STOP_AFTER_UPDATES is not None and STOP_AFTER_UPDATES <= 0:
    raise ValueError("STOP_AFTER_UPDATES phải dương hoặc None")

if FID_NUM_GENERATED < 10 or FID_NUM_GENERATED % 10:
    raise ValueError("FID_NUM_GENERATED phải chia hết cho 10 để cân bằng classes")


In [ ]:
import copy
import pprint
from configs.multitrace_cifar10_22m import config as base_config, scale_training_budget

cfg = copy.deepcopy(base_config)
cfg.update(scale_training_budget(BATCH_SIZE, ACCUMULATION_STEPS))
TRAIN_STEPS = cfg["n_steps"]
cfg.update(num_workers=NUM_WORKERS, data_root=str(VOLUME_ROOT / "data/cifar10"),
           log_step=LOG_EVERY, sample_step=SAMPLE_EVERY,
           checkpoint_step=CHECKPOINT_EVERY, keep_last_checkpoints=KEEP_LAST_CHECKPOINTS,
           mixed_precision="bf16" if torch.cuda.is_bf16_supported() else "fp16",
           source_manifest=SOURCE_MANIFEST,
           monitoring=dict(enabled=True, tensorboard=True, save_best=True, validate_raw=True,
               plot_every=PLOT_EVERY, validation_every=VALIDATION_EVERY,
               validation_batches=VALIDATION_BATCHES, sample_n_per_class=2),
           evaluation=dict(num_generated=FID_NUM_GENERATED, batch_size=FID_BATCH_SIZE,
                           nfes=[1, 2, 3, 4, 5], seed=2026))
if not torch.cuda.is_bf16_supported():
    cfg["multi_trace_imf"]["jvp_precision"] = "fp32"
# JSON normalization keeps resume comparisons stable for list/tuple settings.
cfg = json.loads(json.dumps(cfg))
config_dir = VOLUME_ROOT / "configs"
config_dir.mkdir(exist_ok=True)
CONFIG_PATH = config_dir / f"{RUN_NAME}.py"
CONFIG_PATH.write_text("config = " + pprint.pformat(cfg, sort_dicts=False) + "\n")
child_env = os.environ.copy()
child_env.update(PYTHONUNBUFFERED="1", CUDA_VISIBLE_DEVICES="0", OMP_NUM_THREADS="2",
                 MPLCONFIGDIR="/tmp/imf-matplotlib", TORCH_HOME=str(VOLUME_ROOT / "cache/torch"))
subprocess.run([sys.executable, "train.py", "--config", str(CONFIG_PATH), "--check-model"],
               cwd=SOURCE_DIR, env=child_env, check=True)
print("Microbatch / accumulation / effective batch:", BATCH_SIZE, ACCUMULATION_STEPS,
      BATCH_SIZE * ACCUMULATION_STEPS)
print("Updates / warmup / EMA:", TRAIN_STEPS, cfg["warmup_steps"], cfg["ema_decay"])
print("Run:", RUN_DIR)


In [ ]:
# Đúng model 22M trên GPU, fake data để kiểm tra memory/JVP/backward.
# Tắt diagnostics trong smoke để không lẫn ảnh/log thử vào run thật.
smoke_cfg = copy.deepcopy(cfg)
smoke_cfg["monitoring"] = {"enabled": False}
with tempfile.TemporaryDirectory(prefix="trace-imf-gpu-smoke-") as smoke_dir:
    smoke_config = Path(smoke_dir) / "config.py"
    smoke_config.write_text("config = " + pprint.pformat(smoke_cfg) + "\n")
    subprocess.run([sys.executable, "train.py", "--config", str(smoke_config),
                    "--fake-data", "--steps", "3", "--batch-size", "2",
                    "--num-workers", "0", "--output-dir", smoke_dir],
                   cwd=SOURCE_DIR, env=child_env, check=True)
print("GPU smoke passed at microbatch=2; training microbatch memory is measured in the pilot run")


## Đọc diagnostics trong lúc train

- **Progress bar**: global step, step/s, ETA giờ; `tqdm` trong trainer cũng
  ghi tốc độ và thời gian còn lại. ETA dùng tốc độ thực tế, bao gồm overhead
  đánh giá/vẽ/lưu, và có thể dao động mạnh ở đầu run.
- **Dashboard live**: train edge/diagonal/total loss, held-out loss,
  global gradient trước/sau clip, clipping fraction, LR, JVP RMS,
  actual update/weight từng nhóm và spread của fixed-noise samples.
- **Gradient heatmap**: RMS trước clip của embedding, từng block và head;
  log có missing/zero gradient tensors. Các gate zero-init có thể làm
  gradient trunk bằng 0 ở bước đầu; đó không tự động là lỗi.
- **Validation**: CIFAR-10 test split, cố định ảnh/t/noise qua các lần eval.
  Test split được dùng như validation để tune; cần evaluation độc lập khi
  báo cáo metric cuối. Loss MSE có noise floor, không cần tiến về 0.
- **Samples cố định**: 4 ảnh/class, cùng noise và nhãn ở mỗi lần sample.
  Spread nhỏ là gợi ý để kiểm tra ảnh; không tự chứng minh mode collapse.
- **Convergence report**: ít nhất 5 eval points, gợi ý plateau nếu cải thiện
  <1% trong cửa sổ đó; ghi early/middle/late theo phần budget đã chạy.
  Đây là heuristic để đọc cùng gradient/LR/ảnh; không tự early-stop và
  không chứng minh hội tụ. Warmup dài, LR nhỏ hoặc regression noise floor
  đều có thể làm đường loss phẳng.
- **Files bền vững**: JSONL, CSV, PNG, TensorBoard events và best validation
  weights trên Volume. Khi NaN/Inf, trainer ghi failure.json kèm tên
  parameter có gradient lỗi, dừng run và không upload.

Nếu train loss giảm nhưng validation xấu đi, kiểm tra generalization;
nếu cả hai phẳng, xem LR, warmup, grad/update ratio và clipping. Nếu val
vẫn cải thiện ở cuối budget, có thể cần experiment dài hơn với RUN_NAME mới.
Best validation weights được lưu riêng với step thực tế, bên cạnh final model.

Multi-Trace: validation có cả raw và EMA loss để phân biệt EMA lag với training plateau;
sample/best weights dùng EMA. Mỗi K=1..5 có loss riêng và sample cùng noise.
Gradient heatmap tách từng encoder/decoder level. Train log có effective_batch_size,
images_seen, images_per_second, EMA updates và số update quan sát cho mỗi K.


## Train → resume → FID → upload

Resume khôi phục raw model, optimizer, scheduler, EMA, RNG và global update.
GPU-resident CIFAR loader khôi phục permutation/cursor; CPU/FakeData loader
bắt đầu shuffle mới. Logs sau checkpoint bị loại. Progress/ETA tính theo
optimizer update, gồm 4 microbatches; EMA và LR chỉ update một lần.

Khi thử 1000 updates xong, xem runtime/ETA và clip_fraction. Đặt
`STOP_AFTER_UPDATES=None`, chạy lại settings và pipeline để tiếp tục;
không cần đổi config hoặc RUN_NAME. Nếu restart kernel, Run all.
Thay đổi architecture/effective batch/image budget cần RUN_NAME mới.
Không chạy hai kernel vào cùng run. Interrupt có thể mất các update sau
checkpoint gần nhất. Nếu OOM, dùng RUN_NAME mới và batch64 × accumulation8.

FID dùng **pytorch-fid 0.3.0**, 50k CIFAR train images không augmentation,
mặc định 10k generated/class-balanced cho mỗi NFE1..5, cùng noise/labels.
Đây là FID10k, không phải FID50k và không so trực tiếp với FID trong notebook
tham chiếu (ImageNet Inception weights khác). Eval có progress/ETA riêng,
mất thêm GPU time; real stats được cache, kết quả từng NFE được lưu để retry.
Nếu upload lỗi, chạy lại pipeline: bỏ qua train và phần FID đã xong.
Chỉ upload khi đã đủ training updates và FID hoàn tất.


In [ ]:
from hub import export_model, upload_model
from monitoring import run_with_dashboard

FINAL_CHECKPOINT = RUN_DIR / "ckpts" / f"step_{TRAIN_STEPS:07d}.pt"
saved_config = RUN_DIR / "config.json"
if saved_config.exists() and json.loads(saved_config.read_text()) != cfg:
    raise ValueError("Run có config khác; giữ cấu hình cũ để resume hoặc chọn RUN_NAME mới")
checkpoints = sorted((RUN_DIR / "ckpts").glob("step_*.pt"))
selected_resume = Path(RESUME_CHECKPOINT) if RESUME_CHECKPOINT else (checkpoints[-1] if checkpoints else None)
if not FINAL_CHECKPOINT.exists():
    command = [sys.executable, "train.py", "--config", str(CONFIG_PATH), "--run-dir", str(RUN_DIR)]
    if STOP_AFTER_UPDATES is not None:
        command += ["--stop-after", str(STOP_AFTER_UPDATES)]
    if selected_resume:
        if not AUTO_RESUME:
            raise RuntimeError("Có checkpoint; bật AUTO_RESUME hoặc đổi RUN_NAME")
        if not selected_resume.resolve().is_relative_to((RUN_DIR / "ckpts").resolve()):
            raise ValueError("Resume checkpoint phải thuộc RUN_DIR/ckpts")
        command += ["--resume", str(selected_resume)]
        print("Resume:", selected_resume.name)
    elif RUN_DIR.exists():
        raise RuntimeError("Run tồn tại nhưng chưa có checkpoint; chọn RUN_NAME mới")
    run_with_dashboard(command, SOURCE_DIR, child_env, RUN_DIR, TRAIN_STEPS,
                       refresh_seconds=REFRESH_SECONDS)
else:
    print("Final checkpoint đã có; bỏ qua train và thử upload")
MODEL_URL = None
EXPORT_DIR = VOLUME_ROOT / "hub_exports" / RUN_NAME
if FINAL_CHECKPOINT.exists():
    subprocess.run([sys.executable, "evaluation.py", "--checkpoint", str(FINAL_CHECKPOINT),
                    "--cache-dir", str(VOLUME_ROOT / "cache/fid")],
                   cwd=SOURCE_DIR, env=child_env, check=True)
    export_model(FINAL_CHECKPOINT, EXPORT_DIR, source_dir=SOURCE_DIR)
    MODEL_URL = upload_model(hub_api, REPO_ID, EXPORT_DIR)
    (RUN_DIR / "hub_upload.json").write_text(json.dumps(
        {"repo_id": REPO_ID, "url": MODEL_URL, "training_steps": TRAIN_STEPS,
         "git_commit": GIT_COMMIT}, indent=2))
    print("Hub commit verified:", MODEL_URL)
else:
    print("Pilot paused with checkpoint; set STOP_AFTER_UPDATES=None and rerun settings + pipeline to resume")


In [ ]:
from IPython.display import display, Image, Markdown

if MODEL_URL:
    display(Markdown(f"EMA model đã upload: [{REPO_ID}]({MODEL_URL})"))
for name in ("dashboard.png", "gradient_heatmap.png", "runtime.png", "multitrace.png"):
    path = RUN_DIR / "diagnostics" / name
    if path.exists():
        display(Image(filename=str(path)))
report = RUN_DIR / "diagnostics/convergence.json"
if report.exists():
    print(json.dumps(json.loads(report.read_text()), indent=2))
preview = EXPORT_DIR / "sample_grid.png"
if preview.exists():
    display(Image(filename=str(preview)))
print("Latest checkpoint:", sorted((RUN_DIR / "ckpts").glob("step_*.pt"))[-1])
print("TensorBoard logs:", RUN_DIR / "tensorboard")
print("Best weights + metrics:", RUN_DIR / "best_validation.safetensors", RUN_DIR / "best_validation.json")
print("Stop kernel khi kết thúc để ngừng sử dụng GPU.")

fid_path = RUN_DIR / "fid.json"
if fid_path.exists():
    print(fid_path.read_text())


## Inference / TensorBoard

Hugging Face nhận final EMA `model.safetensors`, config, inference source,
best EMA validation weights, FID protocol/results, samples và diagnostics.
Optimizer checkpoints ở lại Volume. Download repo bằng snapshot_download
rồi cài requirements.txt. Dùng `sample.py --model-dir ... --nfe 1` hoặc
`--nfe 5`; `--weights best_validation.safetensors` chọn best snapshot.
TensorBoard: `tensorboard --logdir /path/to/download/tensorboard`.
`diagnostics/multitrace.png` so loss theo K; từng đường có số quan sát khác
nhau khi train, validation dùng data/noise/interval cố định và EMA.
